In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Carregar os dados
df = pd.read_excel('../dataset/dataset_velocidade_v2.xlsx')

In [3]:
# Criar feature categórica baseada nos dois primeiros dígitos da velocidade
df['vel_str'] = df['velocidade'].apply(lambda x: str(x).split('.')[1][:2])  # '0.0813' -> '08'
# Criar target
df['target'] = (df['vel_str'] != '08').astype(int)  # 0 = normal, 1 = anomalia

In [ ]:
#df.head()
# --- Diagnóstico rápido ---
print("Distribuição das classes (target):")
print(df['target'].value_counts(), "\n")
print("Resumo das velocidades:")
print(df['velocidade'].describe(), "\n")
print("Min/Max:", df['velocidade'].min(), df['velocidade'].max(), "\n")

In [5]:
# Features e target
X = df[['movimento', 'tempo', 'vel_str']]
y = df['target']

# Colunas categóricas e numéricas
cat_features = ['movimento', 'vel_str']
num_features = ['tempo']


In [6]:
from sklearn.preprocessing import OneHotEncoder

# Pré-processamento atualizado
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)  # <--- handle_unknown
    ]
)

# Pipeline com Regressão Logística
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])


In [7]:
# Dividir dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# Treinar modelo
pipeline.fit(X_train, y_train)

# Avaliar
y_pred = pipeline.predict(X_test)
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

Matriz de Confusão:
[[262   0]
 [  0 576]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       262
           1       1.00      1.00      1.00       576

    accuracy                           1.00       838
   macro avg       1.00      1.00      1.00       838
weighted avg       1.00      1.00      1.00       838



In [18]:
import numpy as np

def testar_velocidade_random():
    velocidade_rand = np.round(np.random.uniform(0, 0.2), 5)
    vel_str = str(velocidade_rand).split('.')[1][:2]
    teste = pd.DataFrame({
        'movimento': ['avanco'],  # valor fictício
        'tempo': [1.0],
        'vel_str': [vel_str]
    })
    resultado = pipeline.predict(teste)[0]
    status = "Normal" if resultado == 0 else "Anomalia"
    print(f"Velocidade gerada: {velocidade_rand} → {status} -> {resultado}")

# Testar 10 velocidades aleatórias
for _ in range(10):
    testar_velocidade_random()


Velocidade gerada: 0.08541 → Normal -> 0
Velocidade gerada: 0.05787 → Anomalia -> 1
Velocidade gerada: 0.10441 → Anomalia -> 1
Velocidade gerada: 0.18365 → Anomalia -> 1
Velocidade gerada: 0.16692 → Anomalia -> 1
Velocidade gerada: 0.16154 → Anomalia -> 1
Velocidade gerada: 0.05944 → Anomalia -> 1
Velocidade gerada: 0.01193 → Anomalia -> 1
Velocidade gerada: 0.07996 → Anomalia -> 1
Velocidade gerada: 0.07095 → Anomalia -> 1


In [10]:
# def testar_velocidade_random():
#     import numpy as np
#     velocidade_rand = np.round(np.random.uniform(0, 0.2), 5)
    
#     # Extrair apenas os dois primeiros dígitos após o ponto decimal
#     decimal_str = str(velocidade_rand).split('.')[1][:2]  # '0.08794' -> '08'
#     status = "Normal" if decimal_str == "08" else "Anomalia"
    
#     print(f"Velocidade gerada: {velocidade_rand} → {status}")

# # Testar várias vezes
# for _ in range(10):
#     testar_velocidade_random()
